# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [3]:
# Task 1
# YOUR CODE HERE
import pandas as pd
import plotly.express as px

df = pd.read_csv('../data/global_energy_mix.csv')

source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

# Filter to fossil only
fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# Find most fossil-dependent country and region for the title
top_country = fossil.groupby('Country')['TWh'].sum().idxmax()
top_region  = fossil.groupby('Region')['TWh'].sum().idxmax()

# CVD-safe colours per fossil source
COLOR_MAP = {
    'Coal':        '#E63946',   # red
    'Oil':         '#F4A261',   # orange
    'Natural Gas': '#457B9D',   # blue
}

# ── Treemap ───────────────────────────────────────────────────────────────────
fig = px.treemap(
    fossil,
    path=['Region', 'Country', 'Source'],   # Region → Country → Source hierarchy
    values='TWh',
    color='Source',
    color_discrete_map={
        **COLOR_MAP,
        # Grey out parent nodes (Region and Country levels)
        '(?)': '#CCCCCC'        # px.treemap uses '(?)' for parent node colour
    },
    custom_data=['TWh', 'Source']
)

# Show TWh in labels, no percentages
fig.update_traces(
    texttemplate='%{label}<br>%{value:.0f} TWh',
    textfont=dict(size=12, family='Arial'),
    hovertemplate=(
        '<b>%{label}</b><br>'
        'TWh: %{value:,.0f}<br>'
        'Parent: %{parent}<extra></extra>'
    ),
    marker=dict(
        pad=dict(t=20, l=5, r=5, b=5)
    )
)

fig.update_layout(
    title=dict(
        text=(
            f"<b>{top_country} and the {top_region} region dominate global fossil fuel consumption</b><br>"
            "<sup>Fossil fuel energy (TWh) by Region → Country → Source · "
            "<span style='color:#E63946'>■ Coal</span>  "
            "<span style='color:#F4A261'>■ Oil</span>  "
            "<span style='color:#457B9D'>■ Natural Gas</span></sup>"
        ),
        font=dict(size=15, family='Arial'),
        x=0.5, xanchor='center'
    ),
    font=dict(family='Arial', size=12),
    margin=dict(l=20, r=20, t=110, b=20),
    height=680, width=1050
)


fig.show()


## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [4]:
# Task 2
# YOUR CODE HERE
import plotly.express as px

tips = px.data.tips()

# Aggregate total bill per day → time → smoker group
agg = tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()

# Insight: find day+time combo with most spending
top = agg.groupby(['day', 'time'])['total_bill'].sum().idxmax()
top_day, top_time = top[0], top[1]

# CVD-safe blue/orange for smoker status
COLOR_MAP = {
    'Yes': '#F4A261',   # orange — smoker
    'No':  '#457B9D',   # blue   — non-smoker
    '(?)': '#CCCCCC'    # grey   — parent nodes (day and time levels)
}

fig = px.sunburst(
    agg,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map=COLOR_MAP,
)

fig.update_traces(
    texttemplate='%{label}<br>%{percentParent:.0%}',   # percent parent as required
    textfont=dict(size=12, family='Arial'),
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Total bill: $%{value:,.2f}<br>'
        'Share of parent: %{percentParent:.1%}<extra></extra>'
    ),
    insidetextorientation='radial',
    leaf=dict(opacity=0.9),
)

fig.update_layout(
    title=dict(
        text=(
            f"<b>{top_day} {top_time} drives the most restaurant spending</b><br>"
            "<sup>Total bill ($) by Day → Meal time → Smoker status · "
            "<span style='color:#F4A261'>■ Smoker</span>  "
            "<span style='color:#457B9D'>■ Non-smoker</span></sup>"
        ),
        font=dict(size=15, family='Arial'),
        x=0.5, xanchor='center'
    ),
    font=dict(family='Arial', size=12),
    margin=dict(l=20, r=20, t=110, b=20),
    height=650, width=750
)


fig.show()


## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [2]:
# Task 3 — charts
# YOUR CODE HERE
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/global_energy_mix.csv')

source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

# Filter and aggregate
lc = df[df['Source_Type'] == 'Low-carbon'].groupby('Country')['TWh'].sum().reset_index()
lc = lc.sort_values('TWh', ascending=False).reset_index(drop=True)

# Dummy root node for single-level treemap
lc['All'] = 'Low-carbon'

top_country = lc.iloc[0]['Country']
top_twh     = lc.iloc[0]['TWh']

# ── Chart 1: Treemap ──────────────────────────────────────────────────────────
fig_tree = px.treemap(
    lc,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    custom_data=['TWh']
)

fig_tree.update_traces(
    texttemplate='%{label}<br>%{value:.0f} TWh',
    textfont=dict(size=11, family='Arial'),
    hovertemplate='<b>%{label}</b><br>TWh: %{value:,.0f}<extra></extra>',
    marker=dict(pad=dict(t=20, l=5, r=5, b=5))
)

fig_tree.update_coloraxes(
    colorbar=dict(title='TWh', tickformat=',')
)

fig_tree.update_layout(
    title=dict(
        text=(
            f"<b>{top_country} leads global low-carbon energy production</b><br>"
            "<sup>Total low-carbon TWh (Nuclear + Hydro) per country</sup>"
        ),
        font=dict(size=15, family='Arial'),
        x=0.5, xanchor='center'
    ),
    font=dict(family='Arial', size=12),
    margin=dict(l=20, r=20, t=100, b=20),
    height=620, width=1000
)


fig_tree.show()

# ── Chart 2: Horizontal bar chart ────────────────────────────────────────────
# Top 25 countries for readability
lc_bar = lc.head(25).sort_values('TWh', ascending=True)   # ascending = tallest bar at top

fig_bar = go.Figure()

fig_bar.add_trace(go.Bar(
    x=lc_bar['TWh'],
    y=lc_bar['Country'],
    orientation='h',
    marker=dict(
        color=lc_bar['TWh'],
        colorscale='Blues',
        showscale=False,
        line=dict(width=0)
    ),
    text=lc_bar['TWh'].apply(lambda v: f'{v:,.0f} TWh'),
    textposition='outside',
    textfont=dict(size=10, family='Arial'),
    hovertemplate='<b>%{y}</b><br>TWh: %{x:,.0f}<extra></extra>'
))

fig_bar.update_layout(
    title=dict(
        text=(
            f"<b>{top_country} generates {top_twh:,.0f} TWh from low-carbon sources — more than any other country</b><br>"
            "<sup>Top 25 countries · Nuclear + Hydro combined · sorted by total TWh</sup>"
        ),
        font=dict(size=15, family='Arial'),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Total Low-carbon TWh',
        showgrid=True, gridcolor='#EEEEEE',
        tickformat=','
    ),
    yaxis=dict(
        title='',
        showgrid=False,
        tickfont=dict(size=11)
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=120, r=100, t=110, b=60),
    height=700, width=950
)


fig_bar.show()